In [22]:
import random
import pandas as pd
import datetime
import csv

Initialise global variables for services, normal and anomalous messages, N (for looping), anomaly ration(10% of the logs) and the start date.

In [23]:
SERVICES = [
    "payment-service",
    "auth-service",
    "api-gateway",
    "inventory-service",
    "notification-service"
]

NORMAL_MESSAGES = [
    "Request processed successfully",
    "User authenticated",
    "Cache hit for key",
    "Database query executed in {}ms",
    "Health check passed",
    "Session created for user",
    "Order status updated",
]

ANOMALY_MESSAGES = [
    "Connection timeout after {}ms",
    "Database connection pool exhausted",
    "Unhandled exception in request pipeline",
    "Service unreachable after 3 retries",
    "Memory usage exceeded threshold",
]

N = 10000

ANOMALY_RATIO = 0.1

START_DATE = datetime.datetime.now() - datetime.timedelta(days=30)

Generating a dict in case of anomaly or normal log

In [24]:
def generate_entry(is_anomaly):
    entry = {}
    offset = random.randint(1,2592000)
    timestamp = START_DATE + datetime.timedelta(seconds=offset)
    service = random.choice(SERVICES)
    if is_anomaly:
        level = random.choice(["ERROR", "CRITICAL"])
        message = random.choice(ANOMALY_MESSAGES)
        response_time_ms = random.randint(2000,9000)
        status_code=random.choice(["502","503","500"])
    else:
        level = random.choice(["INFO","WARNING"])
        message=random.choice(NORMAL_MESSAGES)
        response_time_ms = random.randint(50,500)
        status_code=random.choice(["200","201"])
    
    message = message.format(response_time_ms)
    
    entry['timestamp'] = timestamp
    entry['level'] = level
    entry['service'] = service
    entry['message'] = message
    entry['response_time_ms'] = response_time_ms
    entry['status_code'] = status_code
    entry['is_anomaly'] = is_anomaly
    return entry

Appending entries into a list, sorting the list based on timestamp, writing it into a csv file

In [26]:
def main():
    entries = []
    for i in range(N):
        is_anomaly = random.random() < ANOMALY_RATIO
        entries.append(generate_entry(is_anomaly))
    entries.sort(key=lambda x: x['timestamp'])
    with open('../data/logs.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['timestamp','level','service','message','response_time_ms','status_code','is_anomaly'])
        writer.writeheader()
        writer.writerows(entries)

if __name__ == "__main__":
    main()